# Quick Pretrain Test - SWE-Agent MoE
Minimal pretraining run to verify the full pipeline works end-to-end.
After this, run `full_pipeline.ipynb` for the complete training.

In [ ]:
# @title 1. Setup
import os, sys, subprocess
from pathlib import Path

# Mount drive
from google.colab import drive
drive.mount('/content/drive')

# Clone repo
if not Path('/content/model-training-pipeline').exists():
    !cp -r /content/drive/MyDrive/model-training-pipeline /content/
sys.path.insert(0, '/content/model-training-pipeline')

# Install minimal deps
!pip install -q torch transformers accelerate datasets sentencepiece wandb protobuf

# Check GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB' if torch.cuda.is_available() else '')

In [ ]:
# @title 2. Load Minimal Model (for quick test)
from model.architecture import MoEForCausalLM, MoEConfig
from configs.model_config import MoEModelConfig
from transformers import AutoTokenizer

model_config = MoEModelConfig()
hf_config = MoEConfig(
    vocab_size=model_config.vocab_size,
    hidden_size=256,  # tiny for testing
    intermediate_size=1024,
    num_hidden_layers=4,
    num_attention_heads=4,
    num_kv_heads=2,
    num_experts=4,
    num_experts_per_tok=1,
    max_position_embeddings=4096,
)

tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-3B')
tokenizer.pad_token = tokenizer.eos_token
model = MoEForCausalLM(hf_config)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f'Model initialized: {num_params/1e6:.1f}M params (test variant)')
print(f'Activated per token: ~{(4 * hf_config.num_experts_per_tok) / hf_config.num_experts * 100:.0f}% of params')

In [ ]:
# @title 3. Quick Pretraining (100 steps)
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup
from data.pretraining_dataset import create_pretraining_dataloader
import wandb

wandb.init(project='swe-agent-moe-test', config={'phase': 'quick_test'})

dataloader = create_pretraining_dataloader(
    tokenizer=tokenizer, batch_size=1, seq_length=2048, subset=0.01,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)
scheduler = get_cosine_schedule_with_warmup(optimizer, 10, 100)
scaler = torch.cuda.amp.GradScaler()

model.train()
for step, batch in enumerate(dataloader):
    if step >= 100:
        break
    batch = {k: v.to(device) for k, v in batch.items()}

    with torch.cuda.amp.autocast(dtype=torch.bfloat16):
        outputs = model(**batch)
        loss = outputs.loss

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad()
    scheduler.step()

    if step % 10 == 0:
        print(f'Step {step}: loss={loss.item():.4f}')
        wandb.log({'loss': loss.item(), 'step': step})

wandb.finish()
print('Quick pretrain test complete!')

In [ ]:
# @title 4. Run Env Tasks (quick eval)
from rl.environments import SWEEnvironment

env = SWEEnvironment()
model.eval()

for i, task in enumerate(env.tasks[:3]):
    prompt = task.get_prompt()
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=256,
                                 temperature=0.7, top_p=0.95,
                                 pad_token_id=tokenizer.pad_token_id)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    score = task.score_response(response)
    print(f'Task {i+1}: score={score:.2f}')
    print(f'  Response: {response[:100]}...\n')